# Scaled CNN
A copy of `2_scaling_up`, adapted to use a slightly deeper CNN with weight-standardized convolutional layers while keeping the rest of the notebook flow close to the original.


In [24]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron, SpikingGammaOutput
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import compute_linspace_bucket_decay_rates, SpikeConfig
from spikinggamma.dendrite.synapse.conv2d import WSConv2dPerNeuron

THIS_DIR = Path.cwd()
if THIS_DIR.name != "deep_nets":
    THIS_DIR = Path("examples/deep_nets").resolve()
EXAMPLES_DIR = THIS_DIR.parent
REPO_ROOT = EXAMPLES_DIR.parent
for path in (THIS_DIR, EXAMPLES_DIR, REPO_ROOT):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(device)


cuda:1


## Loading the data
We keep the same DVS Gesture loading pattern as before, but use the local helper from this folder.


In [25]:
from load_dvsgesture import get_dvsgesture_dataset
train_dataset, test_dataset, input_shape, n_outputs, T = get_dvsgesture_dataset(timestep_size=12000)
print(input_shape, n_outputs, T)


(2, 32, 32) 11 500


## Building the network
This version stays close to the original notebook structure, but makes the CNN a bit deeper and uses weight-standardized convolutional layers instead of conv-side layer normalization.


In [34]:
# Hyperparameters
n_buckets = 10
decay_factor = 0.1
spike_config = SpikeConfig(
    spike_threshold=0.3
)
conv_decay_rate = torch.tensor([0.9])
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_buckets, decay_factor)

class SpikingGammaNet(nn.Module):
    def __init__(self, 
                 input_shape, 
                 n_outputs,
                 n_buckets,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()
        output_shape_layer1 = (64, 30, 30)
        output_shape_layer2 = (128, 13, 13)
        output_shape_layer3 = (128, 11, 11)
        input_shape_layer4 = (128*5*5,)
        layer4_shape_out = (n_outputs,)

        self.conv1 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(conv_decay_rate, spike_config),
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer1,
                n_buckets=1,
                synaptic_function=WSConv2dPerNeuron(input_shape[0], 64, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction()
        )
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(conv_decay_rate, spike_config),
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer2,
                n_buckets=1,
                synaptic_function=WSConv2dPerNeuron(64, 128, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction()
        )

        self.conv3 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(conv_decay_rate, spike_config),
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer3,
                n_buckets=1,
                synaptic_function=WSConv2dPerNeuron(128, 128, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction()
        )
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(conv_decay_rate, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer4_shape_out,
                n_buckets=1,
                synaptic_function=LinearPerNeuron(input_shape_layer4[0], layer4_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction()
        )

    def reset(self):
        self.conv1.reset()
        self.conv2.reset()
        self.conv3.reset()
        self.fc.reset()

    def detach(self):
        self.conv1.detach()
        self.conv2.detach()
        self.conv3.detach()
        self.fc.detach()

    def forward(self, s_in):
        y_conv1 = self.conv1(s_in)
        y_pool1 = self.pool1(y_conv1)
        y_conv2 = self.conv2(y_pool1)
        y_conv3 = self.conv3(y_conv2)
        y_pool3 = self.pool3(y_conv3)
        y_pool3 = y_pool3.flatten(start_dim=1)
        y_fc = self.fc(y_pool3)
        return y_fc

## Training the network

In [35]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaNet(input_shape, n_outputs, n_buckets, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()


In [37]:
model.train()
n_epochs = 1

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        total_loss = 0
        for t in range(data.shape[0]):
            y = model(data[t])
            loss = loss_fn(y, labels) * 1/data.shape[0]
            loss.backward()
            total_loss += loss.item()

        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 2.3979: 100%|██████████| 34/34 [01:56<00:00,  3.42s/it]


Remember that the first epoch may be slower because of tonic's `DiskCachedDataset` that is enabled by default.

## Testing

In [38]:
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        s_list = [] 
        for t in range(data.shape[0]):
            y = model(data[t])
            s_list.append(y)
        s = torch.stack(s_list, dim=0)
        predictions = s.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 3/3 [00:08<00:00,  3.00s/it]

Test Accuracy: 6.06%
